# 16.2 - Model Serving

Status: VERIFIED

## What Are We Solving?

Training a model is only half the job. Model serving is the bridge between a `.pkl` file and a production prediction. We need to serialize the model, load it efficiently, preprocess inputs the same way as training, and return predictions at scale.

## Mental Model

Serving is like a restaurant kitchen: the trained model is your recipe, preprocessing is your mise en place, and the prediction endpoint is the pass window. Everything must be prepared identically every time.

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

# --- Train ---
X, y = make_classification(n_samples=500, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train_scaled, y_train)
print(f"Train accuracy: {model.score(X_train_scaled, y_train):.3f}")
print(f"Test accuracy:  {model.score(X_test_scaled, y_test):.3f}")


Train accuracy: 0.998
Test accuracy:  0.960


## Save Model + Preprocessor Together

In [2]:
import matplotlib
matplotlib.use('Agg')
import joblib

# Bundle model and scaler into a single artifact
artifact = {
    'model': model,
    'scaler': scaler,
    'feature_names': [f'feat_{i}' for i in range(5)],
    'model_version': '1.0.0',
}
artifact_path = 'serving_demo_artifact.pkl'
joblib.dump(artifact, artifact_path)
print(f"Saved artifact: {artifact_path} ({os.path.getsize(artifact_path)} bytes)")


Saved artifact: serving_demo_artifact.pkl (191250 bytes)


## Load and Predict (Simulated Serving)

In [3]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import joblib

# --- Simulate a fresh server process loading the artifact ---
loaded = joblib.load('serving_demo_artifact.pkl')
serv_model = loaded['model']
serv_scaler = loaded['scaler']
version = loaded['model_version']
print(f"Loaded model version: {version}")

# --- Simulate incoming request ---
raw_input = np.array([[1.2, -0.5, 0.3, 2.1, -1.0]])
preprocessed = serv_scaler.transform(raw_input)
prediction = serv_model.predict(preprocessed)[0]
probabilities = serv_model.predict_proba(preprocessed)[0]

print(f"Raw input:      {raw_input[0]}")
print(f"Preprocessed:   {preprocessed[0].round(3)}")
print(f"Prediction:     {prediction}")
print(f"Probabilities:  {probabilities.round(3)}")
print(f"Confidence:     {max(probabilities):.3f}")


Loaded model version: 1.0.0


Raw input:      [ 1.2 -0.5  0.3  2.1 -1. ]
Preprocessed:   [ 0.821 -0.577  1.62   2.408 -0.733]
Prediction:     0
Probabilities:  [0.54 0.46]
Confidence:     0.540


## Serving Patterns

| Pattern | Use Case | Complexity |
|---------|----------|------------|
| **Synchronous** | Low-latency, single prediction | Low |
| **Batch** | Nightly scoring, large CSVs | Medium |
| **Async queue** | High throughput, non-urgent | High |
| **Streaming** | Real-time data pipelines | High |

In [4]:
import matplotlib
matplotlib.use('Agg')

# Demonstrate batch prediction pattern
import numpy as np

batch_sizes = [1, 10, 50, 100, 500]
print("Batch prediction simulation:")
for bs in batch_sizes:
    X_batch = np.random.randn(bs, 5)
    X_scaled = serv_scaler.transform(X_batch)
    preds = serv_model.predict(X_scaled)
    print(f"  Batch size {bs:>4d}: {preds.sum()} positive predictions")


Batch prediction simulation:
  Batch size    1: 1 positive predictions
  Batch size   10: 6 positive predictions


  Batch size   50: 37 positive predictions
  Batch size  100: 75 positive predictions


  Batch size  500: 364 positive predictions


In [5]:
# Cleanup
import os
if os.path.exists('serving_demo_artifact.pkl'):
    os.remove('serving_demo_artifact.pkl')
print('VERIFICATION PASSED: Phase 16.2 complete')


VERIFICATION PASSED: Phase 16.2 complete
